In [1]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

In [2]:
#大模型
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-7B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

In [3]:
df_dise_2_conc = pd.read_csv("./disease_2_conclusion.csv",keep_default_na=False)
df_dise_2_keyword = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)

In [42]:
df_dise_2_conc[df_dise_2_conc["file_name"]=="肺结节.txt"].head()

,disease_name,file_name,conclusion
867,肺结节,肺结节.txt,"{'诊断': '诊断', '病情现状': '结节大小', '': '发现时间（距离第一次影像..."
868,肺结节,肺结节.txt,"{'诊断': '已接受手术，有明确诊断', '病情现状': '——', '': '——', ..."
869,肺结节,肺结节.txt,"{'诊断': '已接受活检，有明确诊断', '病情现状': '——', '': '——', ..."
870,肺结节,肺结节.txt,"{'诊断': '无明确诊断，只见于当前影像，无既往影像可比较', '病情现状': '——',..."
871,肺结节,肺结节.txt,"{'诊断': '无明确诊断，只见于当前影像，无既往影像可比较', '病情现状': '——',..."


In [4]:
for idx,row in df_dise_2_conc.iterrows():
    if row["file_name"]=="肺结节.txt":
        df_dise_2_conc.loc[idx,"disease_name"] = "肺结节"


In [8]:
df_dise_2_conc_v2 = df_dise_2_conc.drop(104,axis=0,inplace=False)
df_dise_2_conc_v2[df_dise_2_conc_v2["disease_name"]=="视网膜病的病因未明，正在进行检查"].head()

,disease_name,file_name,conclusion


In [9]:
df_dise_2_conc_v2.to_csv("./utils/disease_2_conclusion_v2.csv",index=False)

In [7]:
df_dise_2_conc[df_dise_2_conc["disease_name"]=="视网膜病的病因未明，正在进行检查"].head()

,disease_name,file_name,conclusion
104,视网膜病的病因未明，正在进行检查,五官科.txt,"{'疾病名称': '视网膜病的病因未明，正在进行检查', '诊断': '视网膜病的病因未明，..."


In [5]:
disease_name = set([word for word in df_dise_2_conc["disease_name"].tolist() if word!="视网膜病的病因未明，正在进行检查"])
print(len(disease_name))
print(disease_name)

227
{'继发于慢性病的贫血', '戊型肝炎', '脊柱裂', '癫痫', '结核病', '胰腺炎', '新生儿缺氧缺血性脑病', '视网膜中央动脉血栓症', '心内膜炎', '新生儿溶血病', '心肌炎', '急性肾病综合征（急性肾小球肾炎、感染后肾小球肾炎）', '纵膈肿瘤', '呼吸衰竭', '雅库氏关节病', '植物神经功能紊乱', '耳鸣', '腮腺癌', '肺水肿', '四肢骨折', '胃息肉', '肺结节', '前列腺炎', '鼻炎', '腰椎间盘突出', '乳腺增生', '血红蛋白尿', 'EB病毒感染', '便潜血', '带状疱疹', '脾脏创伤', '多发性硬化', '风湿热', '肾积水', '支气管扩张', '慢性间质性肾炎（慢性肾小管间质性肾炎）', '雷诺综合征', '头痛', '白内障', '股骨头坏死', '气管炎/支气管炎', '肾病综合征（慢性）', '流行性出血热', '胃炎', '视网膜血管病变', '皮肌炎', '肾下垂', '先天性肾畸形/单肾', '附件', '痔疮', '缺铁性贫血', '肛瘘', '子宫出血', '梅毒', '胸膜炎', '扁桃体炎', '烧伤', '乳腺炎、脓肿', '肺动静脉瘘', '重症肌无力', '巨幼细胞性贫血', '色素性视网膜炎', '视网膜血管变性', '腺样体肥大', '丙型肝炎', '扩张性心肌病', '肺纤维化', '子宫内膜增厚（内膜厚度超过10mm）', '甲状腺炎', '主动脉瓣疾病', '焦虑症', '视网膜中央静脉血栓形成', '甲状腺手术', '颈椎病', '脑损伤', '子宫脱垂', '腰肌劳损', '特发性血小板减少紫癜', '腰椎滑脱', '宫颈上皮内瘤变\n', '甲亢', '病理分类： 乳头状癌、滤泡性癌（55岁以下）', '肝性脑病', '肠吸收不良', '慢性肾小球肾炎', '颅咽管瘤', '乳腺假体植入', '视网膜出血', '急性细菌性肾盂肾炎、急性肾盂肾炎', '面神经炎\n（面瘫）', 'HPV感染', '肝血管瘤', '糖尿病前期', '手术', '宫颈癌', '肺炎\n（不包括新冠肺炎）', '血吸虫病', '泌尿系感染', '甲状腺结节', '（下肢）静脉曲张', '类风湿性关节炎', '乳腺结节、囊肿、占位、异常回声', '精神分裂症'

In [80]:
df_dise_2_keyword_kl = df_dise_2_keyword[df_dise_2_keyword["KL_match_disease"].apply(lambda x:x!="")]
print(df_dise_2_keyword_kl.shape)
df_dise_2_keyword_kl[df_dise_2_keyword_kl["disease_n"]=="子宫内膜息肉"].head()
# df_dise_2_keyword_kl.head()


(230, 14)


,def_name,HBZS_def,HBZS_did,disease_n,disease_list,kwords,CI,MI,ADB,Life,ZZ_match_disease,RZ_match_disease,MZ_match_disease,KL_match_disease
492,ZiGNM_XR,ZiGXR,104,子宫内膜息肉,['子宫内膜息肉'],"[['子宫', '宫腔'], ['内膜息肉', '息肉', '高回声']]",子宫内膜恶性肿瘤（包括原位癌）及其复发和转移,子宫内膜息肉,,,zigongzhongliu,zigongneimozhongliu,7881,493


In [70]:

disease_n = set([word for word in df_dise_2_keyword_kl["disease_n"].tolist() if word != ""])
print(len(disease_n))
print(disease_n)

230
{'气胸', '糖耐量异常', '急性肾小球肾炎', '宫颈肿瘤', '二尖瓣反流', '病态窦房结综合征', '儿童乳腺发育', '肾病综合征', '肝血管瘤', '风湿性关节炎', '滋养细胞肿瘤', '焦虑障碍', '肠梗阻', '前列腺炎', '幽门螺旋杆菌感染', '视网膜变性', '主动脉瓣狭窄', '系统性红斑狼疮', '紫癜性肾炎', '颅咽管瘤', '肾下垂', '重症肌无力', '卵巢切除手术', '子宫内膜癌', '结节病', '心包积液', '巨结肠', '乳腺结节', '心肌梗塞', '视网膜出血', '膀胱炎', '宫颈炎', '三叉神经痛', '肾动脉狭窄', '肾炎', '乳腺炎', '血吸虫病', '肝性脑病', '烧伤', '溶血病', '肌营养不良症', '肝囊肿', '动脉栓塞', '支气管扩张症', '主动脉瓣关闭不全', '青光眼', '戊型肝炎', '肛瘘', '子宫肌瘤', '格林巴利综合征', '子宫肉瘤', '肺动静脉瘘', '心肌桥', '腮腺癌', '乙型肝炎', '子宫切除手术', '梗阻性睡眠呼吸暂停', '慢性支气管炎', '脊柱裂', '胆囊切除', '心率不齐', '退行性改变', '宫颈上皮内瘤变', '鼻炎', '肺纤维化', '脑出血', '肺动脉瓣关闭不全', '白内障', '空腹血糖受损', '慢性丙型肝炎', '骨髓增生异常综合征', '房性期前收缩', '梅毒', '肺炎', '缺血缺氧性脑病', '肺水肿', '视网膜色素变性', '气管炎', '肺囊肿', '神经系统肿瘤', '尿道炎', '子宫内膜增生', '流行性乙型脑炎', '脑膜炎', '更年期综合征', '肺或支气管良性肿瘤', '肺结节病', '慢性胰腺炎', '抑郁症', '头痛', '胸膜间皮瘤', 'II型糖尿病', '心肌损害', '心绞痛', '甲状腺结节', '缺铁性贫血', '腰椎滑脱', '子宫内膜异位症', '臂丛神经痛', '雷诺综合征', '脑卒中', '颅脑损伤', '声带结节', '先天性肾畸形', '干眼症', '脂肪肝', '糖尿病性视网膜病变', '卵巢癌', '血红蛋白尿', '酒精性脂肪肝', '肺气肿', '高血压性视网膜病变', '中耳炎', '呼吸衰竭', '输尿管结石'

In [ ]:
if "zi"

In [46]:
#bad key:视网膜病的病因未明，正在进行检查,诊断，
df_dise_2_conc[df_dise_2_conc["disease_name"]=="肺结节"].head()

,disease_name,file_name,conclusion
867,肺结节,肺结节.txt,"{'诊断': '诊断', '病情现状': '结节大小', '': '发现时间（距离第一次影像..."
868,肺结节,肺结节.txt,"{'诊断': '已接受手术，有明确诊断', '病情现状': '——', '': '——', ..."
869,肺结节,肺结节.txt,"{'诊断': '已接受活检，有明确诊断', '病情现状': '——', '': '——', ..."
870,肺结节,肺结节.txt,"{'诊断': '无明确诊断，只见于当前影像，无既往影像可比较', '病情现状': '——',..."
871,肺结节,肺结节.txt,"{'诊断': '无明确诊断，只见于当前影像，无既往影像可比较', '病情现状': '——',..."


In [72]:
all_have = disease_name & disease_n
print(len(all_have))
print(all_have)
disease_name_only = disease_name - all_have
print(len(disease_name_only))
print(disease_name_only)
disease_n_only = disease_n - all_have
print(len(disease_n_only))
print(disease_n_only)

104
{'肝血管瘤', '风湿性关节炎', '前列腺炎', '肠梗阻', '系统性红斑狼疮', '颅咽管瘤', '肾下垂', '重症肌无力', '结节病', '巨结肠', '视网膜出血', '宫颈炎', '三叉神经痛', '肾动脉狭窄', '血吸虫病', '肝性脑病', '烧伤', '肌营养不良症', '肝囊肿', '戊型肝炎', '青光眼', '肛瘘', '子宫肌瘤', '肺动静脉瘘', '腮腺癌', '乙型肝炎', '脊柱裂', '退行性改变', '鼻炎', '肺纤维化', '白内障', '骨髓增生异常综合征', '梅毒', '肺水肿', '流行性乙型脑炎', '更年期综合征', '抑郁症', '头痛', '胸膜间皮瘤', '腰椎滑脱', '甲状腺结节', '缺铁性贫血', '子宫内膜异位症', '臂丛神经痛', '雷诺综合征', '血红蛋白尿', '肺气肿', '呼吸衰竭', '中耳炎', '植物神经功能紊乱', '腰肌劳损', '便潜血', '腹膜炎', '风湿热', '肛裂', '鼻中隔偏曲', 'HPV感染', '强直性脊柱炎', '雅库氏关节病', '流行性出血热', '胆脂瘤', '多发性硬化', '癫痫', '心内膜炎', '鼻部假体植入', '精神分裂症', '子宫内膜息肉', '心肌炎', '脾大', '乳腺假体植入', '痔疮', '脾功能亢进', '甲型肝炎', '遗传性出血性毛细血管扩张症', '颅内神经纤维瘤', '消化性溃疡', '阴道壁膨出', '耳鸣', '房间隔缺损', '近视眼手术', '异位妊娠', 'EB病毒感染', '视网膜中央静脉血栓形成', '短暂性脑缺血发作', '股骨头坏死', '缺血性脑血管病', '哮喘', '胸膜炎', '鼻息肉', '新生儿呼吸窘迫综合征', '胆囊息肉', '神经衰弱', '肝硬化', '乳腺增生', '甲状腺炎', '子宫脱垂', '结直肠癌', '干燥综合征', '视神经炎', '扁桃体炎', '室间隔缺损', '中性粒细胞减少症', '类风湿性关节炎', '腮腺良性肿瘤'}
123
{'新生儿房/室间隔缺损', '甲亢', '泌尿系结石（无高血压和肾功能损害）', '1型糖尿病', '各类肾炎，包含其他因素', '新生儿卵圆孔未闭', '颈椎椎管狭窄', '盆腔积液', '心脏肥

# 编辑距离计算疾病相似度

In [29]:
def edit_distance(str1, str2):
    """计算两个字符串的编辑距离

    Args:
        str1: 字符串1
        str2: 字符串2

    Returns:
        int: 编辑距离
    """

    m = len(str1)
    n = len(str2)

    # 初始化二维数组dp，dp[i][j]表示str1[:i]和str2[:j]的编辑距离
    dp = [[i+j for j in range(n+1)] for i in range(m+1)]
    for i in range(1, m+1):
        dp[i][0] = i
    for j in range(1, n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)

    return dp[m][n]

def normalized_similarity(str1, str2):
    distance = edit_distance(str1, str2)
    max_len = max(len(str1), len(str2))
    similarity = 1 - distance / max_len
    return similarity

# 示例用法
str1 = "kitten"
# str2 = "sitting"
str2 = "kitte"

distance = edit_distance(str2, str1)
print("编辑距离:", distance)

distance_nor = normalized_similarity(str1,str2)
print(round(distance_nor,4))


编辑距离: 1
0.8333


# 构建 中再疾病函数对照表的disease_n 到 disease_2_concluson中的disease_name的匹配

In [75]:
all_have = disease_name & disease_n
print(len(all_have))
print(all_have)
disease_name_only = disease_name - all_have
print(len(disease_name_only))
print("disease name only",disease_name_only)
disease_n_only = disease_n - all_have
print(len(disease_n_only))
print("disease n only",disease_n_only)
map_key_2_dise = {}
for key in list(disease_n_only)+["肾结晶"]:
    if key in ["视网膜病的病因未明，正在进行检查","诊断",""]:
        continue
    simil_score = {}
    for key_n in disease_name_only:
        score = normalized_similarity(key,key_n)
        if score == 0.0:
            continue
        simil_score[key_n] = score
    sorted_dict = sorted(simil_score.items(), key=lambda x: x[1], reverse=True)
    result =sorted_dict[:5]
    map_key_2_dise[key] = result
print(map_key_2_dise)

104
{'肝血管瘤', '风湿性关节炎', '前列腺炎', '肠梗阻', '系统性红斑狼疮', '颅咽管瘤', '肾下垂', '重症肌无力', '结节病', '巨结肠', '视网膜出血', '宫颈炎', '三叉神经痛', '肾动脉狭窄', '血吸虫病', '肝性脑病', '烧伤', '肌营养不良症', '肝囊肿', '戊型肝炎', '青光眼', '肛瘘', '子宫肌瘤', '肺动静脉瘘', '腮腺癌', '乙型肝炎', '脊柱裂', '退行性改变', '鼻炎', '肺纤维化', '白内障', '骨髓增生异常综合征', '梅毒', '肺水肿', '流行性乙型脑炎', '更年期综合征', '抑郁症', '头痛', '胸膜间皮瘤', '腰椎滑脱', '甲状腺结节', '缺铁性贫血', '子宫内膜异位症', '臂丛神经痛', '雷诺综合征', '血红蛋白尿', '肺气肿', '呼吸衰竭', '中耳炎', '植物神经功能紊乱', '腰肌劳损', '便潜血', '腹膜炎', '风湿热', '肛裂', '鼻中隔偏曲', 'HPV感染', '强直性脊柱炎', '雅库氏关节病', '流行性出血热', '胆脂瘤', '多发性硬化', '癫痫', '心内膜炎', '鼻部假体植入', '精神分裂症', '子宫内膜息肉', '心肌炎', '脾大', '乳腺假体植入', '痔疮', '脾功能亢进', '甲型肝炎', '遗传性出血性毛细血管扩张症', '颅内神经纤维瘤', '消化性溃疡', '阴道壁膨出', '耳鸣', '房间隔缺损', '近视眼手术', '异位妊娠', 'EB病毒感染', '视网膜中央静脉血栓形成', '短暂性脑缺血发作', '股骨头坏死', '缺血性脑血管病', '哮喘', '胸膜炎', '鼻息肉', '新生儿呼吸窘迫综合征', '胆囊息肉', '神经衰弱', '肝硬化', '乳腺增生', '甲状腺炎', '子宫脱垂', '结直肠癌', '干燥综合征', '视神经炎', '扁桃体炎', '室间隔缺损', '中性粒细胞减少症', '类风湿性关节炎', '腮腺良性肿瘤'}
123
disease name only {'新生儿房/室间隔缺损', '甲亢', '泌尿系结石（无高血压和肾功能损害）', '1型糖尿病', '各类肾炎，包含其他因素', '新生儿卵圆孔未闭', '颈椎椎

In [76]:
with open("./utils/KL_disease_n_2_name_map.json","w",encoding="utf-8") as f:
    json.dump(map_key_2_dise,f,ensure_ascii=False,indent=4)

In [67]:
map_key_2_dise["肾结"]

[('胆结石', 0.6666666666666667),
 ('肝内胆管结石', 0.33333333333333337),
 ('肾积水', 0.33333333333333337),
 ('肺结节', 0.33333333333333337),
 ('肾错构瘤', 0.25)]

In [77]:
#构建疾病及其相关关键词map, df来自中再疾病函数对照表
disease_n_2_list = {}
for idx,row in df_dise_2_keyword_kl.iterrows():
    disease_n = row["disease_n"].strip()
    disease_list = eval(row["disease_list"])
    # print(type(disease_list))
    try:
        if disease_n  not in disease_n_2_list and disease_n!="":
            disease_n_2_list[disease_n] = []
        disease_n_2_list[disease_n].extend(disease_list)
    except Exception as es:
        print(es)
    

In [78]:
with open("./utils/KL_disease_n_2_keyword.json","w",encoding="utf-8")as f:
    json.dump(disease_n_2_list,f,ensure_ascii=False,indent=4)

In [65]:
# 计算疾病相似度，并返回top_n最相近的疾病
def disease_similarity(input_dise,disease_n_2_list,top_n=3):
    disease_similarity_score = {}
    for k_dise,v_diseKey in disease_n_2_list.items():
        disease_similarity_score[k_dise] = max([normalized_similarity(input_dise,keyw) for keyw in v_diseKey])

    sorted_dict = sorted(disease_similarity_score.items(), key=lambda x: x[1], reverse=True)
    result =sorted_dict[:top_n]
    return result
    

In [66]:
query_dise = "双肾结晶"
results_query = disease_similarity(query_dise,disease_n_2_list)
print(results_query)
print(results_query[0][0])
df_dise_2_conc[df_dise_2_conc["disease_name"]==results_query[0][0]].head()

[('肾结石', 0.5), ('乳腺结节', 0.5), ('肾囊肿', 0.33333333333333337)]
肾结石


,disease_name,file_name,conclusion


In [35]:
df = pd.read_csv("./规则引擎对比结果-评点数据.csv",keep_default_na=False)

In [38]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_time_use = []
for idx,row in df.iterrows():
    
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    print("病人基本信息",basic_info)
    print(image_report)
    # if image_report != "":
    #     image_report = json.loads(image_report)
    #     print(type(image_report))
    #     query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    # else:
    #     query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    query = diagnose
    
    
    try:
        s_time = time.time()
        #获得相似疾病name
        max_simi_dise_names = disease_similarity(query,disease_n_2_list)
        #获得相似疾病的结论
        recall_kbs = [ df_dise_2_conc[df_dise_2_conc["disease_name"]==max_simi_dis[0]]["conclusion"].tolist() for max_simi_dis in max_simi_dise_names]
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)
        ans_list = [recall_kb for recall_kb in recall_kbs if len(recall_kb)>0]
        ans_list = ans_list[0]

    except Exception as es:
        print(es)
        ans_list = []

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,output_struct=output_struct)
    # print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)

    results_underwriting.append(result)
    recall_query.append(ans_list)

    

病人基本信息 年龄:34,性别:女性,临床诊断:宫腔内稍高回声团,考虑子宫内膜息肉,影像报告:{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
recall time use: 0.5981664657592773
病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm

In [39]:
sum(recall_time_use)/len(recall_time_use)

0.3065094128251076

In [42]:
df["RAG核保结论"] = results_underwriting
df["recall_query"] = recall_query
df.to_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv",index=False)

In [48]:
len_get_recall = len([rec_res for rec_res in recall_query if len(rec_res)!=0])
print(len_get_recall)
print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")

14
召回率：43.75%


In [4]:
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_FullTextSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_SemanticSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_HybridSearch.csv")
df_full = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv")


# recall_query = df_full["recall_query"].tolist()
# len_get_recall = len([rec_res for rec_res in recall_query if len(eval(rec_res))!=0])
# print(len_get_recall)
# print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")
bad_dise = []
for idx,row in df_full.iterrows():
    if len(eval(row["recall_query"])) == 0:
        bad_dise.append(row["临床诊断（化验项、疾病等以下划线拼接）"])
print(len(bad_dise))
print(bad_dise)

18
['双肾结石(沙粒样)', '脂肪肝（中度）', '双肾结石', '右侧甲状腺下极下方囊性灶，  考虑甲状旁腺来源可能', '双肾结晶', '脂肪肝', '右侧乳腺结节待查', '右侧乳腺结节待查', '左肺上叶前段磨玻璃影', '甲状腺右叶囊肿(ACRTI-RADS1类)', '两肺多发小结节', '右乳囊性结节拟US-BI-RADS2类', '右肺微小结节', '轻度脂肪肝', '双肾结晶', '窦性心律不齐', '双肾结晶', '肺动脉少量反流']


# check 疾病对照表

In [6]:
print(len(disease_n_2_list))

NameError: name 'disease_n_2_list' is not defined

In [ ]:
and_words = set(disease_n_2_list.keys()) & set(df_dise_2_conc["disease_name"].tolist())
print(len(and_words))

111


In [6]:
df_one_dise = df_dise_2_conc[df_dise_2_conc["disease_name"]=="银行"]
df_one_dise["conclusion"].tolist()

[]